# Agent 1 — Market Research AI Agent
**Owner:** AI/ML Engineer (Member 3)

**Goal:** Given a business idea / category keyword, identify high-growth industries using
startup funding trends (Crunchbase data) and economic indicators (World Bank GDP growth).

**Datasets used:**
- `data/startup_objects.csv` (Crunchbase companies)
- `data/startup_funding_rounds.csv` (Crunchbase funding rounds)
- `data/worldbank_indicators.csv` (World Bank GDP growth %)

**Output:** Industry growth %, startup funding trend, recommended sector — returned as JSON
by `agent_market(keyword)`, which will be wrapped by `market_agent.py` for the FastAPI backend.


In [17]:
!pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.2 MB 7.7 MB/s eta 0:00:01
   ------------ --------------------------- 2.6/8.2 MB 6.7 MB/s eta 0:00:01
   ------------------- -------------------- 3.9/8.2 MB 6.4 MB/s eta 0:00:01
   ------------------------- -------------- 5.2/8.2 MB 6.4 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.2 MB 6.3 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.2 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 6.0 MB/s  0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/36.5 MB 5.9 MB/s eta 0:00:07
   -- ------------------------------------- 2.6/36.5 MB 6.3 MB/s eta 0:

In [18]:
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

from sklearn.metrics import mean_absolute_error, r2_score

DATA_DIR = "../../data"
USD_TO_INR = 83.0

## 2. Load Crunchbase company data
We only need a handful of columns from `objects.csv` (it's a large file), so we
read it with `usecols` and keep only rows where `entity_type == 'Company'`.

In [19]:
object_cols = [
    "id",
    "entity_type",
    "category_code",
    "status",
    "founded_at",
    "country_code",
    "funding_rounds",
    "funding_total_usd"
]

companies = pd.read_csv(
    f"{DATA_DIR}/startup_objects.csv",
    usecols=object_cols,
    low_memory=False
)

companies = companies[
    companies["entity_type"] == "Company"
].copy()

companies["category_code"] = companies[
    "category_code"
].fillna("unknown")

companies["funding_total_usd"] = pd.to_numeric(
    companies["funding_total_usd"],
    errors="coerce"
)

companies["founded_at"] = pd.to_datetime(
    companies["founded_at"],
    errors="coerce"
)

print("Companies:", len(companies))

companies.head()



Companies: 196553


,id,entity_type,category_code,status,founded_at,country_code,funding_rounds,funding_total_usd
0,c:1,Company,web,operating,2005-10-17,USA,3,39750000.0
1,c:10,Company,games_video,acquired,NaT,USA,0,0.0
2,c:100,Company,games_video,acquired,NaT,USA,0,0.0
3,c:10000,Company,network_hosting,operating,2008-07-26,NaN,0,0.0
4,c:10001,Company,games_video,operating,2008-07-26,NaN,0,0.0


## 3. Load funding rounds and attach the year of funding

In [20]:
funding_cols = [
    "funding_round_id",
    "object_id",
    "funded_at",
    "raised_amount_usd"
]

funding = pd.read_csv(
    f"{DATA_DIR}/startup_funding_rounds.csv",
    usecols=funding_cols,
    low_memory=False
)

funding["funded_at"] = pd.to_datetime(
    funding["funded_at"],
    errors="coerce"
)

funding["raised_amount_usd"] = pd.to_numeric(
    funding["raised_amount_usd"],
    errors="coerce"
)

funding["year"] = funding["funded_at"].dt.year

print("Funding Rounds:", len(funding))

funding.head()

Funding Rounds: 52928


,funding_round_id,object_id,funded_at,raised_amount_usd,year
0,1,c:4,2006-12-01,8500000.0,2006.0
1,2,c:5,2004-09-01,500000.0,2004.0
2,3,c:5,2005-05-01,12700000.0,2005.0
3,4,c:5,2006-04-01,27500000.0,2006.0
4,5,c:7299,2006-05-01,10500000.0,2006.0


In [21]:
funding = funding.merge(
    companies[
        ["id", "category_code"]
    ],
    left_on="object_id",
    right_on="id",
    how="left"
)

funding = funding.drop(columns=["id"])

funding = funding.dropna(
    subset=[
        "year",
        "category_code",
        "raised_amount_usd"
    ]
)

funding = funding[
    (funding["year"] >= 2000)
    &
    (funding["year"] <= 2013)
]

print(
    "Years:",
    int(funding["year"].min()),
    "-",
    int(funding["year"].max())
)

funding.head()

Years: 2000 - 2013


,funding_round_id,object_id,funded_at,raised_amount_usd,year,category_code
0,1,c:4,2006-12-01,8500000.0,2006.0,news
1,2,c:5,2004-09-01,500000.0,2004.0,social
2,3,c:5,2005-05-01,12700000.0,2005.0,social
3,4,c:5,2006-04-01,27500000.0,2006.0,social
4,5,c:7299,2006-05-01,10500000.0,2006.0,web


## 4. Yearly funding trend per category
This builds a `category_code x year` table of total raised funding, which is the
basis for the growth-rate calculation and the simple trend forecast.

In [22]:
category_year = (
    funding
    .groupby(
        ["category_code", "year"]
    )["raised_amount_usd"]
    .sum()
    .reset_index()
)

pivot = category_year.pivot(
    index="category_code",
    columns="year",
    values="raised_amount_usd"
).fillna(0)

print("Categories:", len(pivot))

pivot.head()


Categories: 43


year,2000.0,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,2010.0,2011.0,2012.0,2013.0
category_code,,,,,,,,,,,,,,
advertising,39004000.0,25362120.0,61152000.0,18000000.0,30720000.0,1.929910e+08,6.281637e+08,1.190301e+09,1.666092e+09,1.059380e+09,1.444315e+09,2.213153e+09,2.029796e+09,1.827358e+09
analytics,0.0,1400000.0,0.0,3000000.0,23900000.0,7.650312e+07,1.485333e+08,2.560865e+08,2.702720e+08,3.926467e+08,7.896008e+08,1.215660e+09,1.497375e+09,2.023334e+09
automotive,0.0,0.0,0.0,0.0,7500000.0,1.311797e+07,6.495100e+07,1.450000e+08,2.610600e+08,1.340020e+09,4.737805e+08,2.628234e+08,5.454154e+08,4.805524e+08
biotech,14000000.0,56218802.0,16096631.0,90809878.0,240652084.0,1.661240e+09,2.625723e+09,4.079968e+09,2.798730e+09,7.344055e+09,9.002365e+09,1.139126e+10,1.055593e+10,1.689843e+10
cleantech,0.0,0.0,6000000.0,5987940.0,28058000.0,1.706972e+08,4.039607e+08,2.063239e+09,6.595386e+09,3.983960e+09,6.771191e+09,6.505720e+09,5.333994e+09,6.750298e+09


## 5. Load GDP Data


In [ ]:
wb = pd.read_csv(
    f"{DATA_DIR}/worldbank_indicators.csv",
    skiprows=4
)

wb = wb[
    wb["Indicator Name"]
    ==
    "GDP growth (annual %)"
]

year_cols = [
    c
    for c in wb.columns
    if c.isdigit()
]

latest_years = sorted(
    year_cols,
    key=int
)[-5:]

global_gdp_growth = (
    wb[latest_years]
    .mean(numeric_only=True)
    .mean()
)

print(
    "Average Global GDP Growth:",
    round(global_gdp_growth, 2),
    "%"
)


{'growth_pct': 176.9, 'forecast_next_year': 8253464021.1}

In [23]:
def to_inr(usd_amount):

    return round(
        usd_amount * USD_TO_INR,
        2
    )


## 6. Forecast Function

In [33]:
def forecast_category(category_code, target_year=2026):

    if category_code not in pivot.index:
        return None

    series = pivot.loc[category_code]

    series = series[series > 0]

    if len(series) < 3:
        return None

    years = np.array(series.index, dtype=float)
    values = np.array(series.values, dtype=float)

    # Smooth using 3-point moving average
    smoothed = pd.Series(values).rolling(
        window=3,
        min_periods=1
    ).mean()

    coeffs = np.polyfit(
        years,
        smoothed,
        1
    )

    forecast = np.polyval(
        coeffs,
        target_year
    )

    forecast = max(
        float(forecast),
        0
    )

    # CAGR
    n_years = years[-1] - years[0]

    cagr = (
        (
            values[-1] / values[0]
        ) ** (1 / n_years) - 1
    ) * 100

    return {

        "growth_pct":
            round(float(cagr), 2),

        "forecast_2026":
            round(forecast, 2)
    }

In [25]:
def calculate_mae(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)

    return np.mean(np.abs(actual - predicted))


def calculate_r2(actual, predicted):

    actual = np.array(actual)
    predicted = np.array(predicted)

    ss_res = np.sum((actual - predicted) ** 2)

    ss_tot = np.sum(
        (actual - np.mean(actual)) ** 2
    )

    if ss_tot == 0:
        return 0

    return 1 - (ss_res / ss_tot)

In [34]:
def evaluate_category(category_code):

    if category_code not in pivot.index:
        return None

    series = pivot.loc[category_code]
    series = series[series > 0]

    if len(series) < 6:
        return {
            "mae": 0,
            "r2": 0
        }

    years = np.array(series.index, dtype=float)
    values = np.array(series.values, dtype=float)

    maes = []

    for split in range(4, len(values)):

        train_years = years[:split]
        train_values = values[:split]

        test_year = years[split]
        actual = values[split]

        coeffs = np.polyfit(
            train_years,
            train_values,
            1
        )

        pred = np.polyval(
            coeffs,
            test_year
        )

        maes.append(
            abs(actual - pred)
        )

    mae = np.mean(maes)

    return {

        "mae":
            round(float(mae), 2),

        "r2":
            "WalkForward"
    }

In [35]:
ALL_CATEGORIES = sorted(
    pivot.index.unique().tolist()
)

KEYWORD_TO_CATEGORY = {

    "food": "hospitality",
    "delivery": "hospitality",
    "restaurant": "hospitality",

    "fintech": "finance",
    "finance": "finance",

    "education": "education",
    "learning": "education",

    "health": "health",
    "fitness": "health",

    "software": "software",

    "app": "mobile",
    "mobile": "mobile",

    "game": "games_video",

    "travel": "travel",

    "shopping": "ecommerce",
    "ecommerce": "ecommerce"
}

In [36]:
def match_category(keyword):

    keyword = keyword.lower()

    for k, cat in KEYWORD_TO_CATEGORY.items():

        if (
            k in keyword
            and
            cat in ALL_CATEGORIES
        ):
            return cat

    for cat in ALL_CATEGORIES:

        if cat in keyword:
            return cat

    return "software"

## 7. `agent_market(keyword)`
Maps a free-text business idea keyword to the closest Crunchbase `category_code`
(simple substring match — replace with embeddings/fuzzy matching for production),
then returns the JSON structure consumed by the Strategy Generation Agent.

In [41]:
def agent_market(keyword):

    category = match_category(keyword)

    stats = forecast_category(category)

    metrics = evaluate_category(category)

    total_funding = float(
        pivot.loc[category].sum()
    )

    num_companies = int(
        companies[
            companies["category_code"]
            ==
            category
        ].shape[0]
    )

    return {

      "agent": "Market Research Agent",

        "input_keyword": keyword,

        "industry_growth_pct":
            stats["growth_pct"],

        "startup_funding_trend_2026_inr":
            to_inr(
                stats["forecast_2026"]
            ),

        "recommended_sector":
            category
    }


## 8. Test the agent

In [42]:
ideas = [
    "food delivery startup",
    "fintech app for small businesses",
]

for idea in ideas:

    result = agent_market(idea)

    print(
        json.dumps(
            result,
            indent=2
        )
    )

    print("-" * 70)


{
  "agent": "Market Research Agent",
  "input_keyword": "food delivery startup",
  "industry_growth_pct": 76.14,
  "startup_funding_trend_2026_inr": 72403749406.6,
  "recommended_sector": "hospitality"
}
----------------------------------------------------------------------
{
  "agent": "Market Research Agent",
  "input_keyword": "fintech app for small businesses",
  "industry_growth_pct": 29.07,
  "startup_funding_trend_2026_inr": 202686978009.75,
  "recommended_sector": "finance"
}
----------------------------------------------------------------------


## 9. Next steps
- Wrap `agent_market()` (plus `match_category`, `growth_and_forecast`, the pre-computed
  `pivot` table and `global_gdp_growth`) into `agent1_market/market_agent.py` and
  `agent1_market/forecasting.py` so the FastAPI backend can call it directly.
- Pre-compute and cache `pivot` / `global_gdp_growth` at startup (as noted in the
  team responsibilities) instead of recomputing them on every request.
- Optionally replace the linear-trend forecast with Prophet or ARIMA for a smoother
  multi-year projection.
